In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" "boto3==1.42.82" "pytz==2026.1.post1" "requests==2.33.1"

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz|requests"

boto3==1.42.82
pytz==2026.1.post1
requests==2.33.1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys
from pprint import pprint

import boto3

In [8]:
# !docker build --no-cache -q -t credit-risk-training:latest -f /app/give_me_some_credit/sagemaker/training/Dockerfile /app/give_me_some_credit/sagemaker/training/

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/



sha256:34886605be2d4fa7e6aba18779fbf798c0925fcea0b214ad74ab8b6ea0b8c385


In [5]:
!aws s3 ls s3://data-lake --recursive

2026-04-04 21:08:43          0 bronze/
2026-04-04 21:08:40    4983329 bronze/give_me_some_credit/2026-04-04/test/cs-test.csv
2026-04-04 21:08:40    7564965 bronze/give_me_some_credit/2026-04-04/train/cs-training.csv
2026-04-04 21:24:01       7409 credit-risk-training-2026-04-04-21-24-01-785/input/code/prepare_model.py
2026-04-04 21:08:46          0 feast/registry/
2026-04-04 21:08:45          0 gold/
2026-04-04 21:08:39    1329743 gold/give_me_some_credit/test_features/ingestion_date=2026-04-04/part-00000-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet
2026-04-04 21:08:39    1322762 gold/give_me_some_credit/test_features/ingestion_date=2026-04-04/part-00001-a10543aa-4ecc-43e2-aefa-2f14a3f8acc6.c000.snappy.parquet
2026-04-04 21:08:39    1662771 gold/give_me_some_credit/train_features/ingestion_date=2026-04-04/part-00000-783fae56-6653-42e2-934f-441042f16fec.c000.snappy.parquet
2026-04-04 21:08:39    1644117 gold/give_me_some_credit/train_features/ingestion_date=2026-04-04/part-0

In [6]:
def get_latest_ingestion_date(bucket, prefix, s3_endpoint=None):
    s3 = boto3.client("s3", endpoint_url=s3_endpoint)

    # We use delimiter='/' to get "folders" inside CommonPrefixes
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter="/")

    dates = []
    for page in pages:
        for obj in page.get("CommonPrefixes", []):
            # Extract 'YYYY-MM-DD' from 'path/ingestion_date=YYYY-MM-DD/'
            folder_name = obj.get("Prefix").split("=")[-1].strip("/")
            dates.append(folder_name)

    if not dates:
        raise ValueError(f"No partitions found in s3://{bucket}/{prefix}")

    # Sorting ensures '2026-03-22' comes after '2026-03-21'
    return sorted(dates)[-1]


# --- Usage in your script ---
S3_ENDPOINT = "http://localstack:4566"
LATEST_DATE = get_latest_ingestion_date(
    bucket="data-lake",
    prefix="gold/give_me_some_credit/train_features/",
    s3_endpoint=S3_ENDPOINT,
)

print(f"Detected Latest Date: {LATEST_DATE}")

Detected Latest Date: 2026-04-04


In [7]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/training/sm_pipeline.py",
        "--mode",
        "local",
        "--ingestion-date",
        LATEST_DATE,
        "--s3-endpoint",
        "http://localstack:4566",
        "--n-trials",
        "3",
        "--auc-threshold",
        "0.85",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker_pipeline_give_me_some_credit:Args: {'mode': 'local', 'ingestion_date': '2026-04-04', 's3_endpoint': 'http://localstack:4566', 's3_bucket': 'data-lake', 'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'give_me_some_credit', 'n_trials': 3, 'auc_threshold': 0.85, 'training_image': 'credit-risk-training:latest', 'network': 'mlops-lab-net', 'aws_region': 'us-east-1'}
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:sagemaker_session:Local SageMaker session ready (bucket=data-lake)
INFO:botocore.credentials:Found credentials in environment variables.


INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for scheduler via: environment_global.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagema

INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.entities:Starting execution for pipeline CreditRiskTrainingPipeline. Execution ID is 02e5daba-921b-4167-b0d5-9507a0bbd621
INFO:sagemaker.local.entities:Starting pipeline step: 'Preprocessing'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overvie

...

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker.local.entities:Pipeline step 'Evaluation' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'CheckAUCThreshold'
INFO:sagemaker.local.entities:Pipeline step 'CheckAUCThreshold' SUCCEEDED.
INFO:sagemaker.local.entities:Pipeline execution 02e5daba-921b-4167-b0d5-9507a0bbd621 SUCCEEDED
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker_pipeline_give_me_some_credit:Pipeline execution started: local
INFO:sagemaker_pipeline_give_me_some_credit:Pipeline step summary:
INFO:sagemaker_pipeline_give_me_some_credit:Could not retrieve step summary: 'str' object has no attribute 'get'
INFO:sagemaker_pipeline_give_me_some_credit:Pipeline complete.


i5iyyvi5xr-algo-1-ore03  | 2026-04-04 21:36:52,758 - tune_step - INFO - Tuning complete.
i5iyyvi5xr-algo-1-ore03 exited with code 0
 Compose Stopping Aborting on container exit...
 Container i5iyyvi5xr-algo-1-ore03 Stopping 
 Container i5iyyvi5xr-algo-1-ore03 Stopped 
time="2026-04-04T21:36:54Z" level=warning msg="/tmp/tmpbts1wprm/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-04-04T21:36:54Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpbts1wprm\".\nSet `external: true` to use an existing network"
 Container k8prznjaf7-algo-1-mc8um Creating 
 Container k8prznjaf7-algo-1-mc8um Created 
Attaching to k8prznjaf7-algo-1-mc8um
 Container k8prznjaf7-algo-1-mc8um Starting 
 Container k8prznjaf7-algo-1-mc8um Started 
k8prznjaf7-algo-1-mc8um  | 2026-04-04 21:36:57,389 - evaluate - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'give


Exit code: 0
